In [1]:
!pip install wandb --quiet

In [14]:
from torch.utils.data import Dataset
import torchaudio
import torch
from torch.utils.data import DataLoader
import os
import random
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import wandb

In [15]:
DATA_DIR = "./datasets/speech_commands/"

In [16]:
class SpeechCommandDataset(Dataset):
    def __init__(self, samples, noise_signals=None, noise_probability=0.8, snr_range=(3,15),sample_rate=16000, n_fft=512, hop_length=160, n_mels=64, target_length=16000):
        self.samples = samples
        self.noise_signals = noise_signals
        self.noise_probability = noise_probability
        self.snr_range = snr_range
        self.target_length = target_length
        self.transform =torchaudio.transforms.MelSpectrogram(
            sample_rate=sample_rate,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels
        )
    def __len__(self):
        return len(self.samples)

    def _pad_or_trim(self, waveform):
        length = waveform.shape[1]

        if length > self.target_length:
            waveform = waveform[:, :self.target_length]
        elif length < self.target_length:
            waveform = torch.nn.functional.pad(waveform,(0, self.target_length - length))
        return waveform

    def _add_noise(self, waveform):
        noise_signal = random.choice(self.noise_signals)
        start = random.randint(0, noise_signal.shape[1] - self.target_length)
        noise_chunk = noise_signal[:, start: start + self.target_length]

        snr_db = random.uniform(*self.snr_range)
        signal_power = waveform.pow(2).mean() + 1e-10
        noise_power = noise_chunk.pow(2).mean() + 1e-10
        scale = torch.sqrt(signal_power / (noise_power * 10 ** (snr_db / 10)))
        return waveform + scale * noise_chunk
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        waveform, _ = torchaudio.load(path)

        waveform = self._pad_or_trim(waveform)

        if self.noise_signals and random.random() < self.noise_probability:
            waveform = self._add_noise(waveform)

        mel = self.transform(waveform)
        log_mel = torchaudio.transforms.AmplitudeToDB()(mel)
        return log_mel, label

In [17]:
all_dirs = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d)) and d != "_background_noise_"
])
label2idx = {name: idx for idx, name in enumerate(all_dirs)}
idx2label = {idx: name for name, idx in label2idx.items()}

In [18]:
noise_signals = []
for f in os.listdir(os.path.join(DATA_DIR, "_background_noise_")):
    if f.endswith(".wav"):
        noise_path = os.path.join(DATA_DIR, "_background_noise_", f)
        noise_signal, _ = torchaudio.load(noise_path)
        noise_signals.append(noise_signal)

In [19]:
train_sample = []
val_sample = []
test_sample = []

with open(os.path.join(DATA_DIR, "validation_list.txt")) as f:
    val_files = set(line.strip() for line in f)

with open(os.path.join(DATA_DIR, "testing_list.txt")) as f:
    test_files = set(line.strip() for line in f)

for label in label2idx.keys():
    label_dir = os.path.join(DATA_DIR, label)
    for f in os.listdir(label_dir):
        if f.endswith(".wav"):
            path = os.path.join(label_dir, f)
            rel_path = f"{label}/{f}"
            if rel_path in val_files:
                val_sample.append((path, label2idx[label]))
            elif rel_path in test_files:
                test_sample.append((path, label2idx[label]))
            else:
                train_sample.append((path, label2idx[label]))

print(f"Train: {len(train_sample)}, Val: {len(val_sample)}, Test: {len(test_sample)}")

train_ds = SpeechCommandDataset(train_sample, noise_signals=noise_signals)
val_ds = SpeechCommandDataset(val_sample)
test_ds = SpeechCommandDataset(test_sample)

Train: 84843, Val: 9981, Test: 11005


In [20]:
class SpeechCommandCNN(nn.Module):
    def __init__(self, num_classes=35):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
        )

        self.classifier = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

In [21]:
train_loader = DataLoader(
    train_ds, 
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [25]:
config = {
    "model": "SpeechCommandCNN",
    "num_classes": len(label2idx),
    "batch_size": 64,
    "lr": 1e-3,
    "optimizer": "AdamW",
    "scheduler": "ReduceLROnPlateau",
    "n_fft": 512,
    "hop_length": 160,
    "n_mels": 64,
    "noise_prob": 0.8,
    "snr_range": (3, 15),
    "epochs": 20,
}
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
wandb.init(project="learning-audio-ml", config=config)

model = SpeechCommandCNN(num_classes=config["num_classes"])
model.to(device)

steps_per_epoch = len(train_loader)
warmup_steps = steps_per_epoch
total_steps = steps_per_epoch * config["epochs"]

optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=1e-3)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.1, total_iters=warmup_steps)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps - warmup_steps)
scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])

scaler = torch.amp.GradScaler()

criterion = nn.CrossEntropyLoss()


wandb.watch(model, log="all", log_freq=100)

In [ ]:
print(next(model.parameters()).device)

cuda:0


In [27]:
best_val_loss = float('inf')
patience_counter = 0
patience = 5

for epoch in range(config["epochs"]):
    model.train()
    total_loss = 0
    correct_train = 0
    total_train = 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        x,y = x.to(device), y.to(device)
        optimizer.zero_grad()
        
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(x)
            loss = criterion(outputs, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        correct_train += (outputs.argmax(dim=1) == y).sum().item()
        total_train += y.size(0)
    
    avg_loss = total_loss / len(train_loader)
    train_acc = correct_train / total_train

    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc="Validation"):
            x,y = x.to(device), y.to(device)

            with torch.amp.autocast(device_type="cuda"):
                outputs = model(x)
                loss = criterion(outputs, y)
            
            val_loss += loss.item()
            correct += (outputs.argmax(dim=1) == y).sum().item()
            total += y.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc = correct / total

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), "best_model.pth")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_loss,
        "train/acc": train_acc,
        "val/loss": avg_val_loss,
        "val/acc": val_acc,
        "lr": optimizer.param_groups[0]["lr"],
    })

    print(f"Epoch {epoch+1} — Train Loss: {avg_loss:.4f}, Train Acc: {train_acc:.4f}, Val Loss: {avg_val_loss:.4f}, Val Acc: {val_acc:.4f}")

Validation: 100%|██████████| 156/156 [00:35<00:00,  4.44it/s]


Epoch 1 — Train Loss: 3.1349, Train Acc: 0.1930, Val Loss: 2.5167, Val Acc: 0.4194


Validation: 100%|██████████| 156/156 [00:27<00:00,  5.73it/s]


Epoch 2 — Train Loss: 2.4119, Train Acc: 0.4213, Val Loss: 1.8134, Val Acc: 0.5805


Validation: 100%|██████████| 156/156 [00:26<00:00,  5.78it/s]


...

Epoch 18 — Train Loss: 0.4980, Train Acc: 0.8585, Val Loss: 0.4647, Val Acc: 0.8687


Validation: 100%|██████████| 156/156 [00:27<00:00,  5.75it/s]


Epoch 19 — Train Loss: 0.4899, Train Acc: 0.8613, Val Loss: 0.4658, Val Acc: 0.8665


Validation: 100%|██████████| 156/156 [00:27<00:00,  5.70it/s]

Epoch 20 — Train Loss: 0.4748, Train Acc: 0.8645, Val Loss: 0.5145, Val Acc: 0.8501


In [28]:
model.eval()
test_loss = 0
correct = 0
total = 0
with torch.no_grad():
    for x, y in tqdm(test_loader, desc="Test"):
        x, y = x.to(device), y.to(device)

        with torch.amp.autocast(device_type="cuda"):
            outputs = model(x)
            loss = criterion(outputs, y)
        test_loss += loss.item()
        correct += (outputs.argmax(dim=1) == y).sum().item()
        total += y.size(0)

test_acc = correct / total
avg_test_loss = test_loss / len(test_loader)

wandb.log({"test/loss": avg_test_loss, "test/acc": test_acc})
print(f"Test Loss: {avg_test_loss:.4f}, Test Acc: {test_acc:.4f} ({correct}/{total})")

wandb.finish()

Test: 100%|██████████| 172/172 [00:38<00:00,  4.43it/s]

Test Loss: 0.5806, Test Acc: 0.8292 (9125/11005)


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
lr,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test/acc,▁
test/loss,▁
train/acc,▁▃▅▆▆▇▇▇▇▇██████████
train/loss,█▆▅▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/acc,▁▃▄▆▅▇▇▇▇▇▇▇▇█▇▇███▇
val/loss,█▆▄▃▃▂▂▂▂▂▁▂▂▁▂▁▁▁▁▁
epoch,20
lr,0.0001
test/acc,0.82917


In [29]:
torch.save(model.state_dict(), "cnn_kws.pth")

tensor([1, 0, 1])
